# BATCHTRACE AI
## Detecting, Investigating and Evaluating Batch Effects in Biological Datasets

This notebook demonstrates missingness auditing, PCA, a simple linear-model batch adjustment, before/after evaluation, and CSV exports using a **synthetic gene-expression dataset**. It is a learning prototype, not a validated clinical or biological analysis.

### Requirements
Install if needed: `!pip install numpy pandas matplotlib scikit-learn`.

In [ ]:
# BATCHTRACE AI — Synthetic gene-expression batch-effect investigation
# Demonstration only: not real biological data.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline

np.random.seed(42)

# 1. Simulate expression data with biological signal and technical batch effects
n_samples, n_genes = 120, 300
samples = [f"S{i+1:03d}" for i in range(n_samples)]
condition = np.array(["Control"]*60 + ["Treatment"]*60)
batch = np.tile(np.repeat(["Batch_A", "Batch_B", "Batch_C"], 20), 2)

X = np.random.normal(0, 1, (n_samples, n_genes))
X[condition == "Treatment", :35] += 1.4
effects = {
    "Batch_A": np.zeros(n_genes),
    "Batch_B": np.r_[np.repeat(1.3, 100), np.zeros(200)],
    "Batch_C": np.r_[np.zeros(100), np.repeat(-1.1, 100), np.zeros(100)]
}
for i, b in enumerate(batch):
    X[i] += effects[b]

X[np.random.random(X.shape) < 0.01] = np.nan
genes = [f"Gene_{i+1:03d}" for i in range(n_genes)]
expr = pd.DataFrame(X, index=samples, columns=genes)
meta = pd.DataFrame({"Condition": condition, "Batch": batch}, index=samples)

print("Expression matrix:", expr.shape)
display(meta.head(), expr.iloc[:5, :8])

# 2. Missingness audit and median imputation
missing = expr.isna().mean()
print("Overall missingness:", round(expr.isna().mean().mean()*100, 2), "%")
plt.figure(figsize=(8, 4))
plt.hist(missing, bins=25)
plt.title("Missing-value fraction across genes")
plt.xlabel("Fraction missing"); plt.ylabel("Number of genes")
plt.tight_layout(); plt.show()

imputer = SimpleImputer(strategy="median")
Ximp = pd.DataFrame(imputer.fit_transform(expr), index=samples, columns=genes)

# 3. PCA before correction
def pca_scores(data):
    scaled = StandardScaler().fit_transform(data)
    pca = PCA(n_components=2, random_state=42)
    return pca.fit_transform(scaled), pca.explained_variance_ratio_.sum()

def plot_pca(scores, labels, title):
    plt.figure(figsize=(7, 5))
    for label in pd.unique(labels):
        mask = np.asarray(labels) == label
        plt.scatter(scores[mask, 0], scores[mask, 1], label=label, alpha=.8)
    plt.title(title); plt.xlabel("PC1"); plt.ylabel("PC2")
    plt.legend(); plt.tight_layout(); plt.show()

before, variance_before = pca_scores(Ximp)
plot_pca(before, meta["Batch"], "PCA before correction — batch")
plot_pca(before, meta["Condition"], "PCA before correction — condition")
print("PC1+PC2 variance before:", round(variance_before*100, 2), "%")

# 4. Educational linear-model batch adjustment.
# Fit expression ~ intercept + condition + batch; subtract estimated batch terms.
# This is not a substitute for validated ComBat/limma workflows.
cond_dm = pd.get_dummies(meta["Condition"], drop_first=True, dtype=float)
batch_dm = pd.get_dummies(meta["Batch"], drop_first=True, dtype=float)
design = pd.concat([
    pd.Series(1.0, index=samples, name="Intercept"), cond_dm, batch_dm
], axis=1)

corrected = pd.DataFrame(index=samples, columns=genes, dtype=float)
for gene in genes:
    beta = np.linalg.lstsq(design.values, Ximp[gene].values, rcond=None)[0]
    batch_part = batch_dm.values @ beta[-batch_dm.shape[1]:]
    corrected[gene] = Ximp[gene].values - batch_part

# 5. PCA after correction
after, variance_after = pca_scores(corrected)
plot_pca(after, meta["Batch"], "PCA after correction — batch")
plot_pca(after, meta["Condition"], "PCA after correction — condition")
print("PC1+PC2 variance after:", round(variance_after*100, 2), "%")

# 6. Quantify clustering and predictability
batch_code = pd.Categorical(meta["Batch"]).codes
condition_code = pd.Categorical(meta["Condition"]).codes
silhouette = pd.DataFrame({
    "Metric": ["Batch silhouette in PCA", "Condition silhouette in PCA"],
    "Before": [silhouette_score(before, batch_code), silhouette_score(before, condition_code)],
    "After": [silhouette_score(after, batch_code), silhouette_score(after, condition_code)]
})
display(silhouette)

def cv_accuracy(data, target):
    model = make_pipeline(
        StandardScaler(),
        RandomForestClassifier(n_estimators=200, random_state=42)
    )
    return cross_val_score(model, data, target, cv=5, scoring="accuracy").mean()

model_comparison = pd.DataFrame({
    "Prediction task": ["Predict batch", "Predict condition"],
    "Before correction": [
        cv_accuracy(Ximp, meta["Batch"]),
        cv_accuracy(Ximp, meta["Condition"])
    ],
    "After correction": [
        cv_accuracy(corrected, meta["Batch"]),
        cv_accuracy(corrected, meta["Condition"])
    ]
})
display(model_comparison)

# 7. Gene variance comparison
plt.figure(figsize=(7, 6))
plt.scatter(Ximp.var(), corrected.var(), alpha=.6)
plt.xlabel("Variance before correction"); plt.ylabel("Variance after correction")
plt.title("Gene-level variance before vs after")
plt.tight_layout(); plt.show()

# 8. Expression heatmap of top variable genes
top = Ximp.var().nlargest(30).index
plt.figure(figsize=(13, 7))
plt.imshow(corrected[top].T, aspect="auto")
plt.colorbar(label="Expression value")
plt.xticks(range(0, n_samples, 10), samples[::10], rotation=45)
plt.yticks(range(len(top)), top)
plt.title("Top variable genes after demonstration correction")
plt.xlabel("Samples"); plt.ylabel("Genes")
plt.tight_layout(); plt.show()

# 9. Export results
sample_summary = pd.DataFrame({
    "Sample": samples, "Condition": condition, "Batch": batch,
    "PC1_before": before[:, 0], "PC2_before": before[:, 1],
    "PC1_after": after[:, 0], "PC2_after": after[:, 1]
})
expr.to_csv("batchtrace_raw_expression.csv")
Ximp.to_csv("batchtrace_imputed_expression.csv")
corrected.to_csv("batchtrace_corrected_expression.csv")
sample_summary.to_csv("batchtrace_sample_summary.csv", index=False)
silhouette.to_csv("batchtrace_silhouette_metrics.csv", index=False)
model_comparison.to_csv("batchtrace_model_comparison.csv", index=False)

print("BATCHTRACE analysis complete. Six CSV files exported.")
print("Scientific caution: synthetic data only; batch correction may remove real biology,")
print("especially when batch and condition are confounded. Validate on real datasets.")


### Research limitations
- Synthetic data only; no biological conclusions can be drawn.
- The correction is an educational linear-model demonstration, not a replacement for ComBat or limma.
- Batch and condition confounding makes effects difficult or impossible to separate.
- For real research, use documented public expression datasets, validated correction methods, preserve biological covariates, and validate that meaningful biological variation remains.